# Conditional Probability & Independence
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Calculate** conditional probability by restricting the sample space to a known event
- **Explain** what it means for two events to be statistically independent
- **Interpret** the multiplication rule and when it simplifies for independent events

> **Tip:** Start by moving the **P(B) slider** to its extremes, notice how P(A | B) changes when the events overlap a lot versus a little.

---
## How we got here

In *01: Basic Probability* we calculated P(A) by counting outcomes in the full sample space. But real data rarely arrives without context. This notebook asks: what if we already know something? When we condition on an event, we shrink the sample space to only the outcomes where that event occurred, and everything changes.

---
## Why this matters for data science

Conditional probability is the engine behind Bayesian classifiers, recommendation systems, and any model that updates beliefs with new evidence. When a fraud-detection model sees that a transaction came from a foreign country, it computes P(fraud | foreign), not P(fraud), and that distinction is often the difference between catching fraud and missing it.

---
## Try it yourself

In [2]:
from scipy.optimize import brentq
from tkh_utils import make_slider, make_dropdown, make_output, display_widget, register_observer

# ── Circle geometry ────────────────────────────────────────────────────────────
_RMAX = 0.34   # r = _RMAX * sqrt(P)

def _int_area(d, r1, r2):
    if d >= r1 + r2: return 0.0
    if d <= abs(r1 - r2): return np.pi * min(r1, r2)**2
    a1 = np.arccos(np.clip((d**2 + r1**2 - r2**2) / (2*d*r1), -1, 1))
    a2 = np.arccos(np.clip((d**2 + r2**2 - r1**2) / (2*d*r2), -1, 1))
    sq = np.sqrt(max((-d+r1+r2)*(d+r1-r2)*(d-r1+r2)*(d+r1+r2), 0))
    return r1**2 * a1 + r2**2 * a2 - 0.5 * sq

def _solve_d(r1, r2, target):
    m = np.pi * min(r1, r2)**2
    if target <= 0: return r1 + r2 + 0.01
    if target >= m - 1e-9: return max(abs(r1 - r2) - 0.005, 1e-4)
    try:
        return brentq(lambda d: _int_area(d, r1, r2) - target,
                      max(abs(r1 - r2) + 1e-9, 1e-9), r1 + r2 - 1e-9)
    except Exception:
        return (r1 + r2) / 2

def _circle(cx, cy, r, n=200):
    t = np.linspace(0, 2*np.pi, n)
    return cx + r*np.cos(t), cy + r*np.sin(t)

def _intersect(cx1, cy1, r1, cx2, cy2, r2):
    d = np.hypot(cx2 - cx1, cy2 - cy1)
    if d >= r1 + r2 - 1e-9 or d < 1e-10: return None, None
    if d <= abs(r1 - r2) + 1e-9:
        r = min(r1, r2)
        cx, cy_c = (cx1, cy1) if r1 <= r2 else (cx2, cy2)
        return _circle(cx, cy_c, r)
    ang = np.arctan2(cy2 - cy1, cx2 - cx1)
    a1  = np.arccos(np.clip((d**2 + r1**2 - r2**2) / (2*d*r1), -1, 1))
    a2  = np.arccos(np.clip((d**2 + r2**2 - r1**2) / (2*d*r2), -1, 1))
    t1  = np.linspace(ang - a1, ang + a1, 80)
    t2  = np.linspace(ang + np.pi + a2, ang + np.pi - a2, 80)
    xp  = np.concatenate([cx1 + r1*np.cos(t1), cx2 + r2*np.cos(t2), [cx1 + r1*np.cos(t1[0])]])
    yp  = np.concatenate([cy1 + r1*np.sin(t1), cy2 + r2*np.sin(t2), [cy1 + r1*np.sin(t1[0])]])
    return xp, yp

# ── Controls ──────────────────────────────────────────────────────────────────
out = make_output()
pa_slider  = make_slider(0.05, 0.90, 0.05, 0.55, "P(A):")
pb_slider  = make_slider(0.05, 0.90, 0.05, 0.45, "P(B):")
pab_slider = make_slider(0.00, 0.45, 0.05, 0.20, "P(A ∩ B):")
mode_drop  = make_dropdown(
    ["P(A|B)  — condition on B", "P(B|A)  — condition on A"],
    "P(A|B)  — condition on B", "Show:",
)

# ── Render ────────────────────────────────────────────────────────────────────
def render(pa, pb, pab, mode):
    r_a = _RMAX * np.sqrt(pa)
    r_b = _RMAX * np.sqrt(pb)
    d   = _solve_d(r_a, r_b, pab * np.pi * _RMAX**2)
    ca, cb, cy = 0.5 - d/2, 0.5 + d/2, 0.5

    p_ab  = pab / pb if pb > 1e-9 else 0.0
    p_ba  = pab / pa if pa > 1e-9 else 0.0
    indep = abs(pab - pa * pb) < 1e-5
    do_ab = "A|B" in mode

    traces = []

    if do_ab:
        # B filled (the conditioning event — new "universe")
        bx, by = _circle(cb, cy, r_b)
        traces.append(go.Scatter(x=bx, y=by, fill="toself",
            fillcolor="rgba(180,215,180,0.52)",
            line=dict(color=PALETTE["secondary"], width=2.5),
            name="B", hoverinfo="skip"))
        ax, ay = _circle(ca, cy, r_a)
        traces.append(go.Scatter(x=ax, y=ay, fill="none",
            line=dict(color=PALETTE["primary"], width=2.5),
            name="A", hoverinfo="skip"))
    else:
        # A filled (condition on A)
        ax, ay = _circle(ca, cy, r_a)
        traces.append(go.Scatter(x=ax, y=ay, fill="toself",
            fillcolor="rgba(76,110,245,0.22)",
            line=dict(color=PALETTE["primary"], width=2.5),
            name="A", hoverinfo="skip"))
        bx, by = _circle(cb, cy, r_b)
        traces.append(go.Scatter(x=bx, y=by, fill="none",
            line=dict(color=PALETTE["secondary"], width=2.5),
            name="B", hoverinfo="skip"))

    # Intersection — darker fill on top
    ix, iy = _intersect(ca, cy, r_a, cb, cy, r_b)
    if ix is not None:
        traces.append(go.Scatter(x=ix, y=iy, fill="toself",
            fillcolor="rgba(70,115,70,0.72)",
            line=dict(width=0), name="A ∩ B", hoverinfo="skip"))
        mid = (ca + cb) / 2
        traces.append(go.Scatter(x=[mid], y=[cy], mode="markers",
            marker=dict(color="black", size=7),
            showlegend=False, hoverinfo="skip"))

    # Annotations: A, B, S labels + arrow to A∩B
    anns = [
        dict(x=ca - r_a*0.55, y=cy + r_a*0.75, text="<i>A</i>",
             showarrow=False, font=dict(size=20, color=PALETTE["primary"]),
             xanchor="center"),
        dict(x=cb + r_b*0.55, y=cy + r_b*0.75, text="<i>B</i>",
             showarrow=False, font=dict(size=20, color=PALETTE["secondary"]),
             xanchor="center"),
        dict(x=1.04, y=1.07, text="<i>S</i>",
             showarrow=False, font=dict(size=16, color=PALETTE["muted"]),
             xanchor="right"),
    ]
    if ix is not None:
        mid = (ca + cb) / 2
        anns.append(dict(
            x=mid, y=cy,
            ax=mid + 0.22, ay=cy - 0.27,
            xref="x", yref="y", axref="x", ayref="y",
            text="<i>A</i> ∩ <i>B</i>",
            showarrow=True, arrowhead=2, arrowsize=1.2, arrowwidth=1.5,
            font=dict(size=13, color="#333"),
        ))

    ev, cnd   = ("A", "B") if do_ab else ("B", "A")
    result    = p_ab  if do_ab else p_ba
    cond_p    = pb    if do_ab else pa
    ref_p     = pa    if do_ab else pb

    if indep:
        title = (f"A ⊥ B  (Independent)  —  P({ev}|{cnd}) = P({ev}) = {ref_p:.3f}")
    else:
        title = (
            f"P(A) = {pa:.2f}   ·   P(B) = {pb:.2f}   ·   P(A∩B) = {pab:.2f}<br>"
            f"P({ev}|{cnd}) = P(A∩B) / P({cnd}) = {pab:.3f} / {cond_p:.3f}"
            f" = <b>{result:.3f}</b>"
        )

    layout = base_layout(title=title)
    layout.update(
        width=560, height=520,
        xaxis=dict(range=[-0.15, 1.15], showgrid=False, zeroline=False,
                   showticklabels=False, showline=False),
        yaxis=dict(range=[-0.15, 1.15], showgrid=False, zeroline=False,
                   showticklabels=False, showline=False,
                   scaleanchor="x", scaleratio=1),
        shapes=[dict(type="rect", x0=-0.05, y0=-0.05, x1=1.05, y1=1.05,
                     fillcolor="rgba(245,245,245,0.50)",
                     line=dict(color=PALETTE["muted"], width=1.5))],
        annotations=anns,
        showlegend=False,
        margin=dict(l=20, r=20, t=95, b=20),
    )
    fig = go.Figure(data=traces, layout=layout)
    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

# ── Observers ─────────────────────────────────────────────────────────────────
def on_change(change):
    pa = pa_slider.value
    pb = pb_slider.value
    lo = round(max(0.0, pa + pb - 1.0), 6)
    hi = round(min(pa, pb), 6)
    hi = max(hi, lo + 0.01)
    if lo < pab_slider.min: pab_slider.min = lo
    if hi > pab_slider.max: pab_slider.max = hi
    pab_slider.value = max(lo, min(hi, pab_slider.value))
    pab_slider.min = lo
    pab_slider.max = hi
    render(pa, pb, pab_slider.value, mode_drop.value)

register_observer([pa_slider, pb_slider, pab_slider, mode_drop], on_change)
display_widget(VBox([pa_slider, pb_slider, pab_slider, mode_drop]), out)
render(pa_slider.value, pb_slider.value, pab_slider.value, mode_drop.value)


---
## What's happening?

Conditional probability asks: **given that B already happened, how likely is A?** We shrink the universe down to only the outcomes where B occurred, then measure how many of those also include A.

| Symbol | What it controls |
|--------|-----------------|
| P(A) | Baseline probability of event A |
| P(B) | Baseline probability of event B |
| P(A ∩ B) | Probability both events happen together |
| P(A \| B) | Probability of A *given* we know B occurred |

$$P(A \mid B) = \frac{P(A \cap B)}{P(B)}$$

### The independence test

Two events are **independent** if knowing one happened tells you nothing about the other. Mathematically:

$$A \perp B \iff P(A \mid B) = P(A) \iff P(A \cap B) = P(A) \cdot P(B)$$

Drag the widget sliders until P(A ∩ B) equals exactly P(A) × P(B), that's the independence point, and you'll see P(A|B) snap back to equal P(A).

---
## Real-world example: Medical test results

Medical testing is where conditional probability moves from abstract to life-or-death. A positive test result doesn't mean you have the disease, it means the probability of having it *given a positive test* has changed.

The chart below shows hypothetical test outcomes for a population of 10,000 people where the disease prevalence is 2%. Notice:

- **Notice:** Most positive tests come from healthy people (false positives), because healthy people vastly outnumber sick ones, even a small false-positive rate produces many false alarms
- **Notice:** The "true positive" bar (sick and tested positive) is smaller than the "false positive" bar, this is a direct consequence of low base rate P(disease)
- **Notice:** P(disease | positive test) is much lower than the test's advertised 95% sensitivity, because sensitivity and the probability you care about are two different conditional probabilities

> **Discussion question:** If the disease prevalence doubled to 4%, how would the ratio of true positives to false positives change? Would P(disease | positive) go up or down?

In [3]:
# ── Realistic medical testing scenario ─────────────────────────────────────────
# Parameters based on typical screening test literature
population  = 10_000
prevalence  = 0.02          # 2% of population has the disease
sensitivity = 0.95          # P(positive | disease)  — true positive rate
specificity = 0.90          # P(negative | no disease) — true negative rate

n_sick    = int(population * prevalence)
n_healthy = population - n_sick

true_pos  = round(n_sick * sensitivity)
false_neg = n_sick - true_pos
false_pos = round(n_healthy * (1 - specificity))
true_neg  = n_healthy - false_pos

p_disease_given_pos = true_pos / (true_pos + false_pos)

categories = ["True Positive<br>(sick, tested +)", "False Positive<br>(healthy, tested +)",
              "True Negative<br>(healthy, tested −)", "False Negative<br>(sick, tested −)"]
counts     = [true_pos, false_pos, true_neg, false_neg]
colors     = [PALETTE["accent"], PALETTE["secondary"], PALETTE["primary"], PALETTE["muted"]]

fig = go.Figure(go.Bar(
    x=categories, y=counts,
    marker_color=colors,
    text=counts, textposition="outside",
))
layout = base_layout(
    title=f"Medical Test Outcomes — 10,000 people, 2% prevalence<br>"
          f"P(disease | positive) = {p_disease_given_pos:.1%}",
    xaxis_title="Test Outcome",
    yaxis_title="Number of People",
)
fig.update_layout(layout)
fig.show()

### Other everyday conditional probability examples

| Situation | Condition B | Event A | P(A) alone | P(A \| B) |
|-----------|------------|---------|------------|----------|
| Email filtering | Message contains "free money" | Email is spam | ~45% | ~97% |
| Weather forecasting | Sky is overcast at 8am | Rain by noon | ~30% | ~65% |
| Credit scoring | Applicant missed 2+ payments | Default on new loan | ~5% | ~40% |
| Recommendation engine | User watched action film | User clicks another action film | ~10% | ~35% |

---
## Key takeaway

> **Conditional probability shrinks the sample space to what you already know; independence means that shrinking changes nothing — the two events have no information about each other.**

---
*Next up: Bayes' Theorem — how to formally flip a conditional probability and update beliefs with evidence*